## Data transformation

#### Imports

In [6]:
import json
import os
import bisect
import datetime
from collections import defaultdict, Counter

os.makedirs("../data/transf", exist_ok=True)

#### 1. Filter AI events + 30-min time bin

In [7]:
with open("../data/interactions.json") as f:
    events = json.load(f)["events"]

def time_bin(ts):
    dt = datetime.datetime.fromtimestamp(ts, tz=datetime.timezone.utc)
    floored = dt.replace(minute=(dt.minute // 30) * 30, second=0, microsecond=0)
    return floored.strftime("%Y-%m-%dT%H:%M:00Z")

ai_events = [
    {**e, "time_bin": time_bin(e["when"])}
    for e in events
    if any(p.startswith("Agent/") for p in e["parties"])
]

with open("../data/transf/ai_events.json", "w") as f:
    json.dump({"events": ai_events}, f, indent=2)

print(f"Total events:       {len(events)}")
print(f"AI-involved events: {len(ai_events)}")
print("Saved → data/transf/ai_events.json")

Total events:       185147
AI-involved events: 99637
Saved → data/transf/ai_events.json


#### 2. Delegation chain reconstruction

In [8]:
# Sort delegation events by time and trace chains:
# if the source of a delegation was recently a target, it extends an existing chain.
CHAIN_WINDOW = 1800  # 30 min

subs = sorted(
    [e for e in ai_events if e["short_name"] == "queue_subordinate_task"],
    key=lambda e: e["when"]
)

pending = {}  # agent -> (chain_id, depth, timestamp)
chain_counter = 0
chain_events = []

for e in subs:
    ags = [p for p in e["parties"] if p.startswith("Agent/")]
    if len(ags) != 2:
        continue
    src, tgt = ags[0], ags[1]
    ts = e["when"]
    ec = dict(e)

    if src in pending and ts - pending[src][2] <= CHAIN_WINDOW:
        cid, depth, _ = pending[src]
        ec["chain_id"] = cid
        ec["chain_depth"] = depth
        pending[tgt] = (cid, depth + 1, ts)
    else:
        chain_counter += 1
        ec["chain_id"] = chain_counter
        ec["chain_depth"] = 1
        pending[tgt] = (chain_counter, 2, ts)

    chain_events.append(ec)

with open("../data/transf/delegation_chains.json", "w") as f:
    json.dump({"events": chain_events}, f, indent=2)

depth_counts = Counter(e["chain_depth"] for e in chain_events)
print(f"Chains detected:  {chain_counter}")
print(f"Depth distribution: { dict(sorted(depth_counts.items())) }")
print("Saved → data/transf/delegation_chains.json")

Chains detected:  1498
Depth distribution: {1: 1498, 2: 170, 3: 14, 4: 5, 5: 1, 6: 1}
Saved → data/transf/delegation_chains.json


#### 3. Triggered-by tagging

In [9]:
# Build per-agent instruction timeline: assign_agent_task = human, queue_subordinate_task = agent.
# For each AI event, the most recent instruction directed at its actor (within 1 hour) is the trigger.
LOOKBACK = 3600  # 1 hour

instructions = defaultdict(list)  # agent -> [(timestamp, "human"|"agent")]

for e in sorted(events, key=lambda e: e["when"]):
    if e["short_name"] == "assign_agent_task":
        person = e["details"].get("person", "")
        if person:
            instructions["Agent/" + person].append((e["when"], "human"))
    elif e["short_name"] == "queue_subordinate_task":
        ags = [p for p in e["parties"] if p.startswith("Agent/")]
        if len(ags) == 2:
            instructions[ags[1]].append((e["when"], "agent"))

triggered_events = []
for e in ai_events:
    ec = dict(e)
    actor = next((p for p in e["parties"] if p.startswith("Agent/")), None)
    triggered_by = "unknown"

    if actor and actor in instructions:
        ts = e["when"]
        instr = instructions[actor]
        ts_list = [t for t, _ in instr]
        idx = bisect.bisect_right(ts_list, ts) - 1
        if idx >= 0 and ts - ts_list[idx] <= LOOKBACK:
            triggered_by = instr[idx][1]

    ec["triggered_by"] = triggered_by
    triggered_events.append(ec)

with open("../data/transf/triggered_events.json", "w") as f:
    json.dump({"events": triggered_events}, f, indent=2)

counts = Counter(e["triggered_by"] for e in triggered_events)
print(f"human:   {counts['human']}")
print(f"agent:   {counts['agent']}")
print(f"unknown: {counts['unknown']}")
print("Saved → data/transf/triggered_events.json")

human:   20685
agent:   10391
unknown: 68561
Saved → data/transf/triggered_events.json


#### 4. Agent session detection

In [10]:
# A new session starts when the same agent has a gap of more than 5 minutes since its last event.
GAP = 300  # 5 minutes

agent_events_map = defaultdict(list)
for e in sorted(ai_events, key=lambda e: e["when"]):
    actor = next((p for p in e["parties"] if p.startswith("Agent/")), None)
    if actor:
        agent_events_map[actor].append(e)

session_events = []
global_session_id = 0

for agent, evts in sorted(agent_events_map.items()):
    session_id = None
    session_pos = 0
    prev_ts = None

    for e in evts:
        ts = e["when"]
        if prev_ts is None or ts - prev_ts > GAP:
            global_session_id += 1
            session_id = global_session_id
            session_pos = 1
        else:
            session_pos += 1

        ec = dict(e)
        ec["session_id"] = session_id
        ec["session_position"] = session_pos
        ec["session_agent"] = agent
        session_events.append(ec)
        prev_ts = ts

with open("../data/transf/agent_sessions.json", "w") as f:
    json.dump({"events": session_events}, f, indent=2)

session_lengths = Counter(e["session_id"] for e in session_events)
lengths = list(session_lengths.values())
print(f"Sessions detected: {global_session_id}")
print(f"Avg session length: {sum(lengths)/len(lengths):.1f} events")
print(f"Max session length: {max(lengths)} events")
print("Saved → data/transf/agent_sessions.json")

Sessions detected: 6247
Avg session length: 15.9 events
Max session length: 22281 events
Saved → data/transf/agent_sessions.json


#### 5. Daily person-level aggregates

In [11]:
agg = defaultdict(int)  # (person, date, event_type) -> count

for e in ai_events:
    day = datetime.datetime.fromtimestamp(e["when"]).date().isoformat()
    for p in e["parties"]:
        if p.startswith("person:"):
            agg[(p, day, e["short_name"])] += 1

daily_records = [
    {"person": person, "date": day, "event_type": etype, "count": count}
    for (person, day, etype), count in sorted(agg.items())
]

with open("../data/transf/daily_aggregates.json", "w") as f:
    json.dump({"records": daily_records}, f, indent=2)

print(f"Records: {len(daily_records)}")
print("Saved → data/transf/daily_aggregates.json")

Records: 2254
Saved → data/transf/daily_aggregates.json
